# Read-only eFUSE and rolling-key architecture

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

eFUSE key storage is persistent; this notebook performs no MMIO and no eFUSE access. The stock AMD XilNvm example main calls a writer and must not be run as a read-only demonstration. A visible register window is not a readable AES key.

The retained combined image uses a **public all-zero root** with AES-GCM key rolling and HSS authentication. Existing eFUSE selection is not provisioning or proof another board has the same state. Fresh packaging uses `python lab.py boot`; keys/private signing state remain local and excluded from Git.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
inspection = json.loads((ROOT / "evidence/boot/bundle/inspection.json").read_text())
display(inspection["combined_signed_encrypted"])
assert inspection["combined_signed_encrypted"]["payloads_match_plain"]
print("Saved independent rolling-chain verification; no new hardware execution")
links = json.loads((ROOT / "docs/presenter/register_links.json").read_text())
display(links["subsections"]["PMCL_EFUSE"])

{'signatures': {'header': {'key_source': 2781070755,
   'plm_offset': 20400,
   'pl_load': 4060086272,
   'pl_bytes': 1327536,
   'pl_encoded': 1354064,
   'plm_bytes': 26048,
   'plm_encoded': 26656,
   'attributes': 789504},
  'ppk_sha256': '37c3a77aa8d423c528bdc140a689a0f99bd1e10f543cea788d90aff923cfa943',
  'spk_sha256': 'b17adea47869aa3629fb00ada6c4ae9cf323282c3d857110b6cc80e300e98825',
  'spk_signature_valid': True,
  'hash_block_signature_valid': True,
  'boot_header_hash_valid': True,
  'signature_bytes': 9620,
  'signature_offset': 10768,
  'hash_block_offset': 10624,
  'plm_sha256': '4f4c05ca015f4c7505493fe1842b64f0294b5df46f44d2cfb34e2cc46fa6d329',
  'pl_sha256': '65317c1bb81c9ceb1d25cf21427ef20cf7dc426caf439c3cc3dcb1a802b1f528',
  'plm_bytes': 26560,
  'pl_bytes': 1348784,
  'chain_chunks': 170},
 'plm_blocks': 7,
 'pl_blocks': 331,
 'payloads_match_plain': True,
 'scope': 'Signed and encrypted image; PUBLIC ZERO AES root, development authentication'}

Saved independent rolling-chain verification; no new hardware execution


{'url': 'https://docs.amd.com/r/en-US/ug1679-spartan-ultrascaleplus-registers/EFUSE-Module',
 'label': 'EFUSE registers · UG1679',
 'kind': 'public-register-section'}

## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).